# 04 · Persistent vector stores and metadata

**Level:** intermediate · **Time:** 40–50 minutes · **Prerequisites:** notebooks 01–03.

You will persist embeddings in Chroma, retrieve only eligible records, inspect distance
scores, and practise insert/update/delete. Every cell can run in a fresh kernel in order.
All documents describe the fictional **Northstar Analytics** company; policies are demonstration data.

**Demo story:** “The index survives a notebook restart, and HR questions should search HR policies.”
Live mode downloads a local embedding model once and makes **one Claude call** for the answer.
Offline mode uses explicitly labelled lexical vectors, still runs a real Chroma database,
and skips Claude. Install the project requirements and select the workshop kernel first.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the repository or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "rag_workshop").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the LangchainPractical project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rag_workshop import (
    Document, load_documents, chunk_documents, TfidfIndex, DenseIndex,
    bm25_search, reciprocal_rank_fusion, context_text, ask_claude,
    rag_answer, show_hits, settings, offline_mode,
)
import pandas as pd
from IPython.display import display, Markdown

print("Mode:", "OFFLINE: Claude calls skipped" if offline_mode() else "LIVE: Claude calls use API credits")
print("Claude model:", settings().model)


## Step 1 · Prepare records before touching the database

A vector store keeps an ID, a vector, text, and metadata for each chunk. The same embedding
model must encode stored chunks and incoming questions. Metadata is structured data used to
select eligible records; it is not a substitute for the text.

**Predict:** will changing an embedding model preserve the meaning of existing vectors?
No—even equally sized vectors from different models are not interchangeable.


In [ ]:
import hashlib
import json
import numpy as np
import chromadb

documents = load_documents()
chunks = chunk_documents(documents, chunk_size=90, overlap=20)
index = DenseIndex(chunks)
print("Embedding backend:", index.backend)
print("Records:", len(chunks), "| vector shape:", index.vectors.shape)
display(pd.DataFrame([dict(id=d.id, **d.metadata) for d in documents]))


## Step 2 · Use a reproducible collection identity

The fingerprint includes the corpus, chunk boundaries, embedding model, backend, vector
dimension, and distance metric. Editing any of those creates a separate collection, so stale
records or incompatible embeddings cannot silently mix with this exercise. Identical reruns
reuse the same collection. Old fingerprinted collections remain on disk until explicitly removed.

Chroma's persistent client writes to `.cache/chroma/`. No hosted embedding API is involved:
we supply every vector ourselves and set `embedding_function=None`.


In [ ]:
identity = {
    "schema": 1, "backend": str(index.backend),
    "embedding_model": settings().embedding_model,
    "dimension": int(index.vectors.shape[1]), "distance": "cosine",
    "chunks": [{"id": d.id, "text": d.text, "metadata": d.metadata} for d in chunks],
}
fingerprint = hashlib.sha256(
    json.dumps(identity, sort_keys=True).encode("utf-8")
).hexdigest()[:16]
collection_name = "workshop04-" + fingerprint
storage_path = ROOT / ".cache" / "chroma"
client = chromadb.PersistentClient(path=str(storage_path))
collection = client.get_or_create_collection(
    name=collection_name,
    embedding_function=None,
    configuration={"hnsw": {"space": "cosine"}},
    metadata={"workshop": "04", "fingerprint": fingerprint},
)
print("Collection:", collection_name)
print("Persistence directory:", storage_path)


## Step 3 · Upsert is safe to repeat with stable IDs

`upsert` inserts missing IDs and replaces existing IDs. Re-running the cell should keep the
same count. IDs identify chunks; a title alone is not a safe unique ID.

This tiny workshop fits into one upsert. Large corpora need batches, ingestion logs, and
a deletion strategy when source files disappear.


In [ ]:
payload = dict(
    ids=[d.id for d in chunks],
    documents=[d.text for d in chunks],
    metadatas=[d.metadata for d in chunks],
    embeddings=np.asarray(index.vectors, dtype=np.float32).tolist(),
)
collection.upsert(**payload)
count_before = collection.count()
collection.upsert(**payload)
assert collection.count() == count_before == len(chunks)
print("Repeated upsert retained", collection.count(), "records.")

reopened = chromadb.PersistentClient(path=str(storage_path)).get_collection(
    name=collection_name, embedding_function=None
)
print("A second client can read", reopened.count(), "persisted records.")


## Step 4 · Inspect cosine distance, then adapt the result

This collection uses **cosine distance**, where smaller values are better. For this metric
only, `similarity = 1 - distance`; the similarity is not a probability or confidence.
Our helper result format uses `score` for similarity and preserves `distance` for inspection.
A query with no eligible records returns an empty list.


In [ ]:
def chroma_search(question, k=3, where=None):
    if k < 1:
        return []
    eligible = collection.get(where=where, include=[])["ids"]
    if not eligible:
        return []
    vector = np.asarray(index.embed_query(question), dtype=np.float32)
    if np.linalg.norm(vector) == 0:
        # A possible lexical-offline query with no known vocabulary.
        return []
    result = collection.query(
        query_embeddings=[vector.tolist()],
        n_results=min(k, len(eligible)), where=where,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {"id": record_id, "text": text, "metadata": metadata,
         "distance": float(distance), "score": 1.0 - float(distance)}
        for record_id, text, metadata, distance in zip(
            result["ids"][0], result["documents"][0],
            result["metadatas"][0], result["distances"][0]
        )
    ]

question = "How many unused leave days can I carry into next year?"
hits = chroma_search(question, k=4)
display(pd.DataFrame([
    {"id": h["id"], "distance ↓": h["distance"], "similarity ↑": h["score"]}
    for h in hits
]))
display(show_hits(hits))


## Step 5 · Apply filters during retrieval

Choose eligible documents **before** selecting the top k. Retrieving three mixed-department
results and filtering afterward can discard useful results below rank three. Here we derive
the HR department name from the actual fixture, avoiding assumptions about its spelling.

Metadata filters demonstrate a retrieval mechanism. In a real application, authorization
must come from a trusted identity layer; users must not choose their own access labels.


In [ ]:
leave_doc = next(d for d in documents if d.id == "benefits_leave")
department_filter = {"department": leave_doc.metadata["department"]}
filtered_hits = chroma_search(question, k=3, where=department_filter)
display(show_hits(filtered_hits))
assert all(h["metadata"]["department"] == leave_doc.metadata["department"]
           for h in filtered_hits)
# $and expresses multiple metadata conditions explicitly.
combined_filter = {"$and": [department_filter,
                   {"visibility": leave_doc.metadata["visibility"]}]}
print("Combined-filter matches:", len(chroma_search(question, where=combined_filter)))
print("Unknown department:", chroma_search(question, where={"department": "missing-demo"}))


## Step 6 · Generate from the filtered evidence

Pause to read the retrieved text before calling Claude. For the bundled policy, expect
**up to 5 days** and the **March 31** deadline. Check that the answer cites retrieved chunk IDs.
The citation must support the claim; an attractive answer alone is not evidence of success.

**API cost:** the next cell makes one Claude request in live mode; a matching cached request
may be reused by the shared helper. Offline mode explicitly skips generation.


In [ ]:
answer = rag_answer(question, filtered_hits, max_tokens=300)
display(Markdown(answer["text"]))
print({key: answer[key] for key in ("input_tokens", "output_tokens", "cached", "offline")})


## Step 7 · Practise create, read, update, delete on one disposable record

Modifying text requires recomputing its vector. Updating only the text would leave the
search index describing an earlier version. The `try/finally` block removes only our temporary
record, including after an error; it never deletes the policy collection.


In [ ]:
demo_id = "workshop04-disposable-record"
original_count = collection.count()
try:
    text_v1 = "Demo policy: support answers questions about annual leave."
    collection.upsert(
        ids=[demo_id], documents=[text_v1],
        embeddings=[index.embed_query(text_v1).tolist()],
        metadatas=[{"department": "Demo", "visibility": "demo"}],
    )
    print("Created:", collection.get(ids=[demo_id])["documents"][0])
    text_v2 = "Demo policy: support answers questions about travel expenses."
    collection.update(ids=[demo_id], documents=[text_v2],
                      embeddings=[index.embed_query(text_v2).tolist()])
    print("Updated:", collection.get(ids=[demo_id])["documents"][0])
finally:
    collection.delete(ids=[demo_id])
assert collection.count() == original_count
print("Temporary record deleted; policy records retained.")


## Step 8 · Change one variable during the demonstration

Compare `k=1` with `k=4` for a question combining leave and remote work. Then try a
product question under the HR filter: a nearest neighbor is not necessarily an answer.
Discuss when to return “insufficient evidence” instead of forcing generation.


In [ ]:
demo_question = "How much paid leave do I get, and how many days can I work remotely?"
for k in (1, 4):
    demo_hits = chroma_search(demo_question, k=k, where=department_filter)
    print("k =", k, "| distinct sources =",
          sorted({h["metadata"]["source_id"] for h in demo_hits}))
    display(show_hits(demo_hits))


## Your turn

1. Retrieve a Team plan question using the product document's department.
2. Assert that every returned document satisfies that department filter.
3. Explain why a collection name should change if chunk size or embedding model changes.

**Presenter checkpoint:** show the collection count, eligible metadata, retrieved text, and
final answer separately. This makes the data path visible to your audience.


In [ ]:
# Sample solution: use the metadata that is present in the actual corpus.
product_doc = next(d for d in documents if d.id == "product_plans")
product_department = product_doc.metadata["department"]
solution_hits = chroma_search("What does the Team plan cost?", k=3,
                              where={"department": product_department})
assert all(h["metadata"]["department"] == product_department for h in solution_hits)
display(show_hits(solution_hits))
print("Fingerprint changes prevent mixing obsolete chunks or incompatible vector spaces.")


## What you can now demonstrate

Persistent storage, repeatable ingestion, filtered nearest-neighbor search, and a complete
record lifecycle. Our corpus is tiny: timing here does not establish production scalability.
Offline lexical vectors validate plumbing, not semantic retrieval quality.

**Next:** notebook 05 expresses the retrieval-to-answer flow with LangChain runnables.

Official references: [Chroma collection configuration](https://docs.trychroma.com/docs/collections/configure),
[metadata filtering](https://docs.trychroma.com/docs/querying-collections/metadata-filtering),
[query and get](https://docs.trychroma.com/docs/querying-collections/query-and-get).
